# DPO Training — Member A, Week 2

Runs from the repo root (this notebook lives at `dpo/dpo_colab.ipynb`).
Designed to be executed via the **Colab VS Code extension** so paths are local — no Drive mount required.

**Prereqs already met:**
- `sft-qwen-0.5b/` LoRA adapter exists at the repo root.
- A T4 (or better) GPU is connected to this kernel.

**Pipeline:** install deps → GPU sanity check → smoke run → full run → quick generation sanity check.

## 1. Install dependencies

In [1]:
%pip install -q "torch>=2.2.0" "transformers>=4.40.0" "datasets>=2.19.0" "peft>=0.10.0" "trl>=0.11.0" "accelerate>=0.29.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.7 MB/s eta 0:00:00:00:0100:01


## 2. GPU sanity check

In [5]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Verify repo layout

In [ ]:
print("lol")

In [ ]:
import os

print(os.getcwd())
print(os.listdir("/content"))
!echo lol

In [7]:
import os
# We expect this notebook to be opened from the repo root via the Colab VS Code extension.
# If not, cd to the repo root.
if not os.path.exists('sft-qwen-0.5b/adapter_config.json'):
    os.chdir('..')
assert os.path.exists('sft-qwen-0.5b/adapter_config.json'), 'SFT adapter missing at sft-qwen-0.5b/'
assert os.path.exists('dpo/train_dpo.py'), 'dpo/train_dpo.py missing'
print('Repo root OK:', os.getcwd())

## 4. Smoke run (~10 min on T4)

Trains 200 steps on ~500 preference pairs. Watch that DPO loss trends down.

In [ ]:
!python -m dpo.train_dpo --fast

## 5. Full run

Trains 1 epoch over 10k preference pairs. Saves the DPO adapter to `checkpoints/dpo-qwen-0.5b/`.

In [ ]:
!python -m dpo.train_dpo

## 6. Generation sanity check

Load the DPO adapter on top of the base model and produce a few completions.
Eyeball for coherence — broken adapter merges produce gibberish.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = 'Qwen/Qwen2.5-0.5B'
DPO_DIR = 'checkpoints/dpo-qwen-0.5b'

tok = AutoTokenizer.from_pretrained(DPO_DIR)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, device_map='cuda')
model = PeftModel.from_pretrained(base, DPO_DIR).eval()

prompts = [
    'Explain photosynthesis in one paragraph.',
    'Write a haiku about reinforcement learning.',
    'List three reasons to use unit tests.',
]
for p in prompts:
    text = tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=160, do_sample=False, pad_token_id=tok.eos_token_id)
    print('Q:', p)
    print('A:', tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
    print('---')